# NLP Assignment — Add-K Smoothing (K = 0.3)

Apply Add-K smoothing with K=0.3 to Unigram, Bigram, Trigram and Quadrigram models and evaluate on 1,000 development and 1,000 test sentences.


## 1. Imports


In [5]:
import re
import math
from collections import Counter
import pyarrow.parquet as pq
import pandas as pd


## 2. Configuration


In [6]:
TRAIN_FILE = "../ASS-4/train.parquet"
DEV_FILE = "../ASS-4/dev.parquet"
TEST_FILE = "../ASS-4/test.parquet"
K = 0.3
print("K =", K)


K = 0.3


## 3. Assignment-1 tokenizer


In [ ]:
TOKEN_PATTERN = re.compile(r"""
    https?://[^\s]+|
    \b[\w.+-]+@[\w.-]+\.\w+\b|
    \d{1,2}[/-]\d{1,2}[/-]\d{2,4}|
    \d+\.\d+|
    [\u0C66-\u0C6F]+|
    \d+|
    [\u0C00-\u0C7F]+|
    [A-Za-z]+|
    \.{3}|
    [.,!?;:()"'\-]
""", re.VERBOSE)

def tokenize(sentence):
    return TOKEN_PATTERN.findall(sentence)


## 4. Load and verify datasets


In [8]:
train_pf = pq.ParquetFile(TRAIN_FILE)
dev_pf = pq.ParquetFile(DEV_FILE)
test_pf = pq.ParquetFile(TEST_FILE)
print("Train:", train_pf.metadata.num_rows)
print("Dev:", dev_pf.metadata.num_rows)
print("Test:", test_pf.metadata.num_rows)
assert train_pf.metadata.num_rows == 998000
assert dev_pf.metadata.num_rows == 1000
assert test_pf.metadata.num_rows == 1000


Train: 998000
Dev: 1000
Test: 1000


## 5. Prepare sentences


In [9]:
def prepare_sentence(sentence):
    return ["<START>"] + tokenize(sentence) + ["<END>"]


## 6. Build vocabulary from training data only


In [10]:
vocab = {"<START>", "<END>", "<UNK>"}
for batch in train_pf.iter_batches(batch_size=10000, columns=["text"]):
    for sentence in batch["text"].to_pylist():
        vocab.update(tokenize(sentence))
V = len(vocab)
print("Vocabulary size V:", V)
token_to_id = {t:i for i,t in enumerate(vocab)}
START_ID = token_to_id["<START>"]
END_ID = token_to_id["<END>"]
UNK_ID = token_to_id["<UNK>"]


Vocabulary size V: 593723


## 7. Build four n-gram counts


In [11]:
unigram_counts = Counter()
bigram_counts = Counter()
trigram_counts = Counter()
quadrigram_counts = Counter()
processed = 0
for batch in train_pf.iter_batches(batch_size=10000, columns=["text"]):
    for sentence in batch["text"].to_pylist():
        ids = [token_to_id.get(t, UNK_ID) for t in prepare_sentence(sentence)]
        unigram_counts.update(ids)
        bigram_counts.update(zip(ids, ids[1:]))
        trigram_counts.update(zip(ids, ids[1:], ids[2:]))
        quadrigram_counts.update(zip(ids, ids[1:], ids[2:], ids[3:]))
        processed += 1
    if processed % 100000 == 0:
        print("Processed", processed)
print("Final processed:", processed)
assert processed == 998000
total_tokens = sum(unigram_counts.values())
print("Unique unigrams:", len(unigram_counts))
print("Unique bigrams:", len(bigram_counts))
print("Unique trigrams:", len(trigram_counts))
print("Unique quadrigrams:", len(quadrigram_counts))


Processed 100000
Processed 200000
Processed 300000
Processed 400000
Processed 500000
Processed 600000
Processed 700000
Processed 800000
Processed 900000
Final processed: 998000
Unique unigrams: 593722
Unique bigrams: 4813387
Unique trigrams: 8244124
Unique quadrigrams: 9197704


## 8. Add-K probability functions


In [12]:
def unigram_add_k_probability(w):
    return (unigram_counts[w] + K) / (total_tokens + K * V)

def bigram_add_k_probability(w1, w2):
    return (bigram_counts[(w1,w2)] + K) / (unigram_counts[w1] + K * V)

def trigram_add_k_probability(w1, w2, w3):
    return (trigram_counts[(w1,w2,w3)] + K) / (bigram_counts[(w1,w2)] + K * V)

def quadrigram_add_k_probability(w1, w2, w3, w4):
    return (quadrigram_counts[(w1,w2,w3,w4)] + K) / (trigram_counts[(w1,w2,w3)] + K * V)


## 9. Convert sentences to IDs


In [13]:
def sentence_to_ids(sentence):
    return [token_to_id.get(t, UNK_ID) for t in prepare_sentence(sentence)]


## 10. Add-K perplexity


In [14]:
def calculate_add_k_perplexity(sentences, model):
    total_log_probability = 0.0
    total_predictions = 0
    for sentence in sentences:
        ids = sentence_to_ids(sentence)
        if model == "unigram":
            for w in ids:
                total_log_probability += math.log(unigram_add_k_probability(w))
                total_predictions += 1
        elif model == "bigram":
            for i in range(1, len(ids)):
                total_log_probability += math.log(bigram_add_k_probability(ids[i-1], ids[i]))
                total_predictions += 1
        elif model == "trigram":
            for i in range(2, len(ids)):
                total_log_probability += math.log(trigram_add_k_probability(ids[i-2], ids[i-1], ids[i]))
                total_predictions += 1
        elif model == "quadrigram":
            for i in range(3, len(ids)):
                total_log_probability += math.log(quadrigram_add_k_probability(ids[i-3], ids[i-2], ids[i-1], ids[i]))
                total_predictions += 1
        else:
            raise ValueError("Invalid model")
    return math.exp(-total_log_probability / total_predictions)


## 11. Load development and test sets


In [15]:
dev_sentences = dev_pf.read_row_group(0)["text"].to_pylist()
test_sentences = test_pf.read_row_group(0)["text"].to_pylist()
assert len(dev_sentences) == 1000
assert len(test_sentences) == 1000
print("Development:", len(dev_sentences))
print("Test:", len(test_sentences))


Development: 1000
Test: 1000


## 12. Development / Validation results


In [16]:
dev_add_k_unigram = calculate_add_k_perplexity(dev_sentences, "unigram")
dev_add_k_bigram = calculate_add_k_perplexity(dev_sentences, "bigram")
dev_add_k_trigram = calculate_add_k_perplexity(dev_sentences, "trigram")
dev_add_k_quadrigram = calculate_add_k_perplexity(dev_sentences, "quadrigram")
print("Unigram:", dev_add_k_unigram)
print("Bigram:", dev_add_k_bigram)
print("Trigram:", dev_add_k_trigram)
print("Quadrigram:", dev_add_k_quadrigram)


Unigram: 3671.434061482684
Bigram: 12981.252223300247
Trigram: 120294.17664909312
Quadrigram: 301374.6526042435


## 13. Test results


In [17]:
test_add_k_unigram = calculate_add_k_perplexity(test_sentences, "unigram")
test_add_k_bigram = calculate_add_k_perplexity(test_sentences, "bigram")
test_add_k_trigram = calculate_add_k_perplexity(test_sentences, "trigram")
test_add_k_quadrigram = calculate_add_k_perplexity(test_sentences, "quadrigram")
print("Unigram:", test_add_k_unigram)
print("Bigram:", test_add_k_bigram)
print("Trigram:", test_add_k_trigram)
print("Quadrigram:", test_add_k_quadrigram)


Unigram: 4022.0497091211246
Bigram: 14591.42648019511
Trigram: 132181.5923501705
Quadrigram: 324671.91022525495


## 14. Final Add-K comparison


In [18]:
add_k_results = pd.DataFrame({
    "Model": ["Unigram", "Bigram", "Trigram", "Quadrigram"],
    "K": [K]*4,
    "Development Perplexity": [dev_add_k_unigram, dev_add_k_bigram, dev_add_k_trigram, dev_add_k_quadrigram],
    "Test Perplexity": [test_add_k_unigram, test_add_k_bigram, test_add_k_trigram, test_add_k_quadrigram]
})
display(add_k_results)
add_k_results.to_csv("add_k_results_k_0.3.csv", index=False)
print("Saved: add_k_results_k_0.3.csv")


,Model,K,Development Perplexity,Test Perplexity
0,Unigram,0.3,3671.434061,4022.049709
1,Bigram,0.3,12981.252223,14591.426480
2,Trigram,0.3,120294.176649,132181.592350
3,Quadrigram,0.3,301374.652604,324671.910225


Saved: add_k_results_k_0.3.csv


## 15. Conclusion

Add-K smoothing with K=0.3 was applied to all four n-gram language models. The models were evaluated on 1,000 development/validation sentences and 1,000 test sentences. Lower perplexity indicates better predictive performance.
